# 🔐 RBA Dataset — Semi-Supervised Anomaly Detection
### Risk Scores & Alerts for Suspicious Login Activities

**Pipeline (Fixed & Leak-Free):**
- ✅ Load `Book1.xlsx` from Google Drive or local upload
- ✅ Proper **80/20 holdout split** *before* any feature engineering
- ✅ Leakage-safe feature engineering (train stats only applied to test)
- ✅ Semi-supervised masking inside train set only — **40% labelled**
- ✅ **6 Models**: Label Propagation · Label Spreading · Self-Training RF · Self-Training Extra Trees · Isolation Forest · One-Class SVM
- ✅ Dynamic Weighted Ensemble Risk Score (weights from labelled val — no test leakage)
- ✅ Accuracy + F1 + Classification Report
- ✅ Alert Generation: LOW / MEDIUM / HIGH / CRITICAL
- ✅ Full Visualisation Dashboard

---
**All models are strictly Semi-Supervised or Unsupervised — no fully supervised classifiers.**

## 📦 Cell 1 — Install & Import Libraries

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib openpyxl

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler, OrdinalEncoder
from sklearn.semi_supervised import LabelPropagation, LabelSpreading, SelfTrainingClassifier
from sklearn.ensemble        import RandomForestClassifier, IsolationForest, ExtraTreesClassifier
from sklearn.svm             import OneClassSVM
from sklearn.metrics         import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve,
    average_precision_score, roc_curve, f1_score,
    accuracy_score,
    recall_score
)
from sklearn.impute      import SimpleImputer
from sklearn.decomposition import PCA

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ All libraries imported!')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')


## 📁 Cell 2 — Load Dataset

In [ ]:
# Option A: Load from Google Drive  (USE_DRIVE = True)
# Option B: Local / Colab upload     (USE_DRIVE = False)
# Option C: Direct path              (set DIRECT_PATH below)

USE_DRIVE   = False
DIRECT_PATH = '/content/Book1.xlsx'

if os.path.exists(DIRECT_PATH):
    FILE_PATH = DIRECT_PATH
    print(f'📂 Using file at: {FILE_PATH}')
elif USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    FILE_PATH = '/content/drive/MyDrive/Book1.xlsx'
else:
    from google.colab import files
    print('Upload your Book1.xlsx file:')
    uploaded  = files.upload()
    FILE_PATH = list(uploaded.keys())[0]

print(f'\n📂 Loading: {FILE_PATH}')
assert os.path.exists(FILE_PATH), f'❌ File not found: {FILE_PATH}'

df_raw = pd.read_excel(FILE_PATH)

TARGET = 'Is Attack IP'
assert TARGET in df_raw.columns, f'❌ Target column "{TARGET}" not found!'
df_raw[TARGET] = df_raw[TARGET].astype(bool).astype(int)

print(f'\n📊 Shape      : {df_raw.shape}')
print(f'📋 Columns    : {list(df_raw.columns)}')
print(f'🎯 Attack rate: {df_raw[TARGET].mean():.2%}  ({df_raw[TARGET].sum()} attacks)')
display(df_raw.head(3))


## 🔍 Cell 3 — Exploratory Data Analysis

In [ ]:
print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'Rows         : {len(df_raw):,}')
print(f'Columns      : {df_raw.shape[1]}')
print(f'Missing vals :\n{df_raw.isnull().sum()[df_raw.isnull().sum() > 0]}')
print()
print('🎯 Label distribution:')
print(df_raw[TARGET].value_counts())

attack_rate = df_raw[TARGET].mean()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('EDA — RBA Dataset', fontsize=14, fontweight='bold')

ax = axes[0]
ax.pie([1 - attack_rate, attack_rate],
       labels=['Normal', 'Attack'], autopct='%1.2f%%',
       colors=['steelblue', 'crimson'], startangle=90)
ax.set_title('Class Balance')

ax = axes[1]
if 'Device Type' in df_raw.columns:
    df_raw['Device Type'].value_counts().plot.bar(ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Device Type Distribution')
    ax.tick_params(axis='x', rotation=30)

ax = axes[2]
if 'Login Successful' in df_raw.columns:
    ct = pd.crosstab(df_raw['Login Successful'].astype(bool),
                     df_raw[TARGET].astype(bool), normalize='index') * 100
    ct.plot.bar(ax=ax, color=['steelblue', 'crimson'], edgecolor='white')
    ax.set_title('Attack Rate by Login Success')
    ax.set_xlabel('Login Successful'); ax.set_ylabel('% of logins')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(['Normal', 'Attack'], fontsize=9)

plt.tight_layout()
plt.show()
print('\n✅ EDA complete!')


## ✂️ Cell 4 — Train / Test Split (BEFORE Feature Engineering)
> **Why first?** Any feature that uses group statistics (e.g. per-user failure rate)
> must be computed on training data only, then applied to test — otherwise test
> information leaks into training and inflates metrics artificially.

In [ ]:
TEST_SIZE   = 0.20
LABEL_RATIO = 0.40   # ✅ 40% of train labelled — enough for strong semi-supervised learning

idx_all = np.arange(len(df_raw))
train_idx, test_idx = train_test_split(
    idx_all,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_raw[TARGET].values
)

df_train_raw = df_raw.iloc[train_idx].copy().reset_index(drop=True)
df_test_raw  = df_raw.iloc[test_idx ].copy().reset_index(drop=True)

print(f'Train set : {len(df_train_raw):,} rows  ({df_train_raw[TARGET].mean():.2%} attack rate)')
print(f'Test set  : {len(df_test_raw):,} rows  ({df_test_raw[TARGET].mean():.2%} attack rate)')
print('\n✅ Split complete — test set is now locked away.')


## ⚙️ Cell 5 — Feature Engineering (Leakage-Safe)
All encoders / statistics are **fit on `df_train_raw` only**, then applied to both sets.

In [ ]:
EXCLUDE = {
    TARGET,
    'Is Account Takeover',
    'index',
    'IP Address',
    'User Agent String',
    'Login Timestamp',
}

RTT_COL         = 'Round-Trip Time [ms]'
RISKY_COUNTRIES = {'RU', 'CN', 'KP', 'IR', 'NG'}

CAT_COLS = [c for c in [
    'Country', 'Region', 'City',
    'OS Name and Version', 'Browser Name and Version', 'Device Type'
] if c in df_raw.columns]


def _temporal(df):
    if 'Login Timestamp' not in df.columns:
        return df
    ts = pd.to_datetime(df['Login Timestamp'], errors='coerce')
    df['hour_of_day'] = ts.dt.hour.fillna(0).astype(int)
    df['day_of_week']  = ts.dt.dayofweek.fillna(0).astype(int)
    df['is_night']     = ((ts.dt.hour >= 22) | (ts.dt.hour <= 5)).astype(int)
    df['is_weekend']   = (ts.dt.dayofweek >= 5).astype(int)
    return df


def engineer(df, fit_objects=None):
    out      = df.copy()
    is_train = (fit_objects is None)
    if is_train:
        fit_objects = {}

    out = _temporal(out)

    if RTT_COL in out.columns:
        out['rtt_log'] = np.log1p(out[RTT_COL].fillna(0).clip(0))
        if is_train:
            fit_objects['rtt_q95'] = out[RTT_COL].quantile(0.95)
        out['rtt_high'] = (out[RTT_COL] > fit_objects['rtt_q95']).astype(int)
    else:
        out['rtt_log']  = 0.0
        out['rtt_high'] = 0

    if 'Login Successful' in out.columns:
        out['login_fail'] = (~out['Login Successful'].astype(bool)).astype(int)
    else:
        out['login_fail'] = 0

    if 'User ID' in out.columns:
        if is_train:
            user_stats = out.groupby('User ID')['login_fail'].agg(
                fail_rate='mean', login_count='count')
            fit_objects['user_stats']    = user_stats
            fit_objects['overall_fail']  = out['login_fail'].mean()
        us = fit_objects['user_stats']
        out['user_fail_rate']   = out['User ID'].map(us['fail_rate']).fillna(fit_objects['overall_fail'])
        out['user_login_count'] = out['User ID'].map(us['login_count']).fillna(1)

    if 'Country' in out.columns:
        out['country_risky'] = out['Country'].isin(RISKY_COUNTRIES).astype(int)

    if 'ASN' in out.columns:
        out['asn_suspicious'] = (
            pd.to_numeric(out['ASN'], errors='coerce').fillna(0) >= 500_000
        ).astype(int)

    present_cats = [c for c in CAT_COLS if c in out.columns]
    if present_cats:
        if is_train:
            enc = OrdinalEncoder(handle_unknown='use_encoded_value',
                                 unknown_value=-1, dtype=np.float64)
            enc.fit(out[present_cats].astype(str))
            fit_objects['ord_enc']  = enc
            fit_objects['cat_cols'] = present_cats
        enc         = fit_objects['ord_enc']
        cat_encoded = enc.transform(out[fit_objects['cat_cols']].astype(str))
        for j, col in enumerate(fit_objects['cat_cols']):
            out[col + '_enc'] = cat_encoded[:, j]

    for col in out.columns:
        if out[col].dtype == bool:
            out[col] = out[col].astype(int)

    to_drop  = [c for c in EXCLUDE   if c in out.columns]
    to_drop += [c for c in CAT_COLS  if c in out.columns]
    out = out.drop(columns=to_drop, errors='ignore')

    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors='coerce')

    return out, fit_objects


y_train = df_train_raw[TARGET].astype(int).values
y_test  = df_test_raw[TARGET].astype(int).values

df_train_fe, FIT = engineer(df_train_raw, fit_objects=None)
df_test_fe,  _   = engineer(df_test_raw,  fit_objects=FIT)

common_cols  = [c for c in df_train_fe.columns if c != TARGET]
for col in common_cols:
    if col not in df_test_fe.columns:
        df_test_fe[col] = 0.0

FEATURE_COLS = [c for c in common_cols if c != TARGET]
X_train_raw  = df_train_fe[FEATURE_COLS].values
X_test_raw   = df_test_fe[FEATURE_COLS].values

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp  = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

print(f'Train feature matrix : {X_train_scaled.shape}')
print(f'Test  feature matrix : {X_test_scaled.shape}')
print(f'Features used        : {FEATURE_COLS}')
print('\n✅ Feature engineering complete — no leakage!')


## 🏷️ Cell 6 — Semi-Supervised Label Masking
> **40%** of training rows are labelled; the rest are masked as `-1` (unlabelled).
> The test set remains completely separate throughout.

In [ ]:
print(f'🏷️  Semi-supervised split — {LABEL_RATIO:.0%} of train data labelled')

y_semi  = np.full(len(y_train), -1, dtype=int)
pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]

print(f'   Train positives (attacks): {len(pos_idx)}')
print(f'   Train negatives (normal) : {len(neg_idx)}')

if len(pos_idx) == 0:
    raise ValueError('❌ No positive samples in training set.')

n_pos = max(2, int(len(pos_idx) * LABEL_RATIO))
n_neg = min(len(neg_idx), n_pos * 9)

lab_pos = np.random.choice(pos_idx, n_pos, replace=False)
lab_neg = np.random.choice(neg_idx, n_neg, replace=False)
lab_idx = np.concatenate([lab_pos, lab_neg])

y_semi[lab_idx] = y_train[lab_idx]

print(f'\n   Labelled positives  : {n_pos}')
print(f'   Labelled negatives  : {n_neg}')
print(f'   Unlabelled rows     : {(y_semi == -1).sum():,}')
print(f'   Total train rows    : {len(y_train):,}')
print(f'\n   Test rows (held-out): {len(y_test):,}  ← never seen during training')
print('\n✅ Label masking ready!')


## 🔢 Cell 7 — PCA (shared by graph-based models)

In [ ]:
print('🔢 Fitting PCA on training set (used by Label Propagation & Label Spreading)...')

n_components = min(15, X_train_scaled.shape[1])
pca = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f'   PCA variance explained : {pca.explained_variance_ratio_.sum():.2%}  ({n_components} components)')
print('✅ PCA ready!')


## 🤖 Cell 8 — Model 1: Label Propagation (Semi-Supervised)

In [ ]:
print('🤖 [Model 1] Training Label Propagation...')

lp_model = LabelPropagation(
    kernel='rbf',    # ✅ rbf avoids disconnected graph nodes (knn caused NaN scores)
    gamma=20,
    max_iter=1000,
    tol=1e-3
)
lp_model.fit(X_train_pca, y_semi)

lp_proba_test  = np.nan_to_num(lp_model.predict_proba(X_test_pca)[:, 1],  nan=0.0)
lp_proba_train = np.nan_to_num(lp_model.predict_proba(X_train_pca)[:, 1], nan=0.0)

lp_auc = roc_auc_score(y_test, lp_proba_test)
lp_ap  = average_precision_score(y_test, lp_proba_test)
print(f'   📈 ROC-AUC       (test): {lp_auc:.4f}')
print(f'   📈 Avg Precision (test): {lp_ap:.4f}')
print('✅ Label Propagation done!')

# ── Per-model metrics (using best_thresh from ensemble — set default 0.5 for individual eval)
_thresh_ind = 0.5
_pred = (lp_proba_test >= _thresh_ind).astype(int)
_acc  = accuracy_score(y_test, _pred)
_rec  = recall_score(y_test, _pred, zero_division=0)
_f1   = f1_score(y_test, _pred, zero_division=0)
_auc  = roc_auc_score(y_test, lp_proba_test)
_ap   = average_precision_score(y_test, lp_proba_test)
print(f'\n   ┌── Individual Model Metrics (threshold=0.5) ──────')
print(f'   │  Accuracy  : {_acc*100:.2f}%')
print(f'   │  Recall    : {_rec:.4f}')
print(f'   │  F1-Score  : {_f1:.4f}')
print(f'   │  ROC-AUC   : {_auc:.4f}')
print(f'   │  Avg Prec  : {_ap:.4f}')
print(f'   └───────────────────────────────────────────────────')


## 🤖 Cell 9 — Model 2: Label Spreading (Semi-Supervised)

In [ ]:
print('🤖 [Model 2] Training Label Spreading...')

ls_model = LabelSpreading(
    kernel='rbf',
    gamma=20,
    alpha=0.2,      # low alpha = trust labels more
    max_iter=1000,
    tol=1e-4,
    n_jobs=-1
)
ls_model.fit(X_train_pca, y_semi)

ls_proba_test  = np.nan_to_num(ls_model.predict_proba(X_test_pca)[:, 1],  nan=0.0)
ls_proba_train = np.nan_to_num(ls_model.predict_proba(X_train_pca)[:, 1], nan=0.0)

ls_auc = roc_auc_score(y_test, ls_proba_test)
ls_ap  = average_precision_score(y_test, ls_proba_test)
print(f'   📈 ROC-AUC       (test): {ls_auc:.4f}')
print(f'   📈 Avg Precision (test): {ls_ap:.4f}')
print('✅ Label Spreading done!')

# ── Per-model metrics (using best_thresh from ensemble — set default 0.5 for individual eval)
_thresh_ind = 0.5
_pred = (ls_proba_test >= _thresh_ind).astype(int)
_acc  = accuracy_score(y_test, _pred)
_rec  = recall_score(y_test, _pred, zero_division=0)
_f1   = f1_score(y_test, _pred, zero_division=0)
_auc  = roc_auc_score(y_test, ls_proba_test)
_ap   = average_precision_score(y_test, ls_proba_test)
print(f'\n   ┌── Individual Model Metrics (threshold=0.5) ──────')
print(f'   │  Accuracy  : {_acc*100:.2f}%')
print(f'   │  Recall    : {_rec:.4f}')
print(f'   │  F1-Score  : {_f1:.4f}')
print(f'   │  ROC-AUC   : {_auc:.4f}')
print(f'   │  Avg Prec  : {_ap:.4f}')
print(f'   └───────────────────────────────────────────────────')


## 🤖 Cell 10 — Model 3: Self-Training Random Forest (Semi-Supervised)

In [ ]:
print('🤖 [Model 3] Training Self-Training Random Forest...')
print('   Iteratively pseudo-labels high-confidence unlabelled rows')

base_rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,       # ✅ full depth trees
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

# ✅ sklearn ≥1.2: parameter is `estimator` (not `base_estimator`)
st_model = SelfTrainingClassifier(
    estimator=base_rf,
    threshold=0.70,
    criterion='threshold',
    max_iter=20,
    verbose=True
)
st_model.fit(X_train_scaled, y_semi)

st_proba_test  = st_model.predict_proba(X_test_scaled)[:, 1]
st_proba_train = st_model.predict_proba(X_train_scaled)[:, 1]

st_auc = roc_auc_score(y_test, st_proba_test)
st_ap  = average_precision_score(y_test, st_proba_test)
print(f'\n   📈 ROC-AUC       (test): {st_auc:.4f}')
print(f'   📈 Avg Precision (test): {st_ap:.4f}')
print('✅ Self-Training RF done!')

# ── Per-model metrics (using best_thresh from ensemble — set default 0.5 for individual eval)
_thresh_ind = 0.5
_pred = (st_proba_test >= _thresh_ind).astype(int)
_acc  = accuracy_score(y_test, _pred)
_rec  = recall_score(y_test, _pred, zero_division=0)
_f1   = f1_score(y_test, _pred, zero_division=0)
_auc  = roc_auc_score(y_test, st_proba_test)
_ap   = average_precision_score(y_test, st_proba_test)
print(f'\n   ┌── Individual Model Metrics (threshold=0.5) ──────')
print(f'   │  Accuracy  : {_acc*100:.2f}%')
print(f'   │  Recall    : {_rec:.4f}')
print(f'   │  F1-Score  : {_f1:.4f}')
print(f'   │  ROC-AUC   : {_auc:.4f}')
print(f'   │  Avg Prec  : {_ap:.4f}')
print(f'   └───────────────────────────────────────────────────')


## 🤖 Cell 11 — Model 4: Self-Training Extra Trees (Semi-Supervised)

In [ ]:
print('🤖 [Model 4] Training Self-Training Extra Trees...')

base_et = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

et_model = SelfTrainingClassifier(
    estimator=base_et,
    threshold=0.70,
    criterion='threshold',
    max_iter=20,
    verbose=False
)
et_model.fit(X_train_scaled, y_semi)

et_proba_test  = et_model.predict_proba(X_test_scaled)[:, 1]
et_proba_train = et_model.predict_proba(X_train_scaled)[:, 1]

et_auc = roc_auc_score(y_test, et_proba_test)
et_ap  = average_precision_score(y_test, et_proba_test)
print(f'   📈 ROC-AUC       (test): {et_auc:.4f}')
print(f'   📈 Avg Precision (test): {et_ap:.4f}')
print('✅ Self-Training Extra Trees done!')

# ── Per-model metrics (using best_thresh from ensemble — set default 0.5 for individual eval)
_thresh_ind = 0.5
_pred = (et_proba_test >= _thresh_ind).astype(int)
_acc  = accuracy_score(y_test, _pred)
_rec  = recall_score(y_test, _pred, zero_division=0)
_f1   = f1_score(y_test, _pred, zero_division=0)
_auc  = roc_auc_score(y_test, et_proba_test)
_ap   = average_precision_score(y_test, et_proba_test)
print(f'\n   ┌── Individual Model Metrics (threshold=0.5) ──────')
print(f'   │  Accuracy  : {_acc*100:.2f}%')
print(f'   │  Recall    : {_rec:.4f}')
print(f'   │  F1-Score  : {_f1:.4f}')
print(f'   │  ROC-AUC   : {_auc:.4f}')
print(f'   │  Avg Prec  : {_ap:.4f}')
print(f'   └───────────────────────────────────────────────────')


## 🤖 Cell 12 — Model 5: Isolation Forest (Unsupervised)

In [ ]:
print('🤖 [Model 5] Training Isolation Forest...')
print('   Unsupervised — uses no labels at all')

iso = IsolationForest(
    n_estimators=300,
    contamination=float(y_train.mean()),
    max_features=1.0,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
iso.fit(X_train_scaled)

iso_raw_train = -iso.decision_function(X_train_scaled)
iso_min, iso_max = iso_raw_train.min(), iso_raw_train.max()
iso_score_train = ((iso_raw_train - iso_min) / (iso_max - iso_min + 1e-9)).clip(0, 1)

iso_raw_test   = -iso.decision_function(X_test_scaled)
iso_score_test = ((iso_raw_test  - iso_min) / (iso_max - iso_min + 1e-9)).clip(0, 1)

iso_auc = roc_auc_score(y_test, iso_score_test)
iso_ap  = average_precision_score(y_test, iso_score_test)
print(f'   📈 ROC-AUC       (test): {iso_auc:.4f}')
print(f'   📈 Avg Precision (test): {iso_ap:.4f}')
print('✅ Isolation Forest done!')

# ── Per-model metrics (using best_thresh from ensemble — set default 0.5 for individual eval)
_thresh_ind = 0.5
_pred = (iso_score_test >= _thresh_ind).astype(int)
_acc  = accuracy_score(y_test, _pred)
_rec  = recall_score(y_test, _pred, zero_division=0)
_f1   = f1_score(y_test, _pred, zero_division=0)
_auc  = roc_auc_score(y_test, iso_score_test)
_ap   = average_precision_score(y_test, iso_score_test)
print(f'\n   ┌── Individual Model Metrics (threshold=0.5) ──────')
print(f'   │  Accuracy  : {_acc*100:.2f}%')
print(f'   │  Recall    : {_rec:.4f}')
print(f'   │  F1-Score  : {_f1:.4f}')
print(f'   │  ROC-AUC   : {_auc:.4f}')
print(f'   │  Avg Prec  : {_ap:.4f}')
print(f'   └───────────────────────────────────────────────────')


## 🤖 Cell 13 — Model 6: One-Class SVM (Unsupervised)

In [ ]:
print('🤖 [Model 6] Training One-Class SVM...')
print('   Trained on normal-only rows — learns the boundary of legitimate logins')

normal_idx_train = np.where(y_train == 0)[0]
X_train_normal   = X_train_scaled[normal_idx_train]

ocsvm = OneClassSVM(kernel='rbf', nu=0.10, gamma='scale')
ocsvm.fit(X_train_normal)

ocsvm_raw_train = -ocsvm.decision_function(X_train_scaled)
ocsvm_min, ocsvm_max = ocsvm_raw_train.min(), ocsvm_raw_train.max()
ocsvm_score_train = ((ocsvm_raw_train - ocsvm_min) / (ocsvm_max - ocsvm_min + 1e-9)).clip(0, 1)

ocsvm_raw_test   = -ocsvm.decision_function(X_test_scaled)
ocsvm_score_test = ((ocsvm_raw_test  - ocsvm_min) / (ocsvm_max - ocsvm_min + 1e-9)).clip(0, 1)

ocsvm_auc = roc_auc_score(y_test, ocsvm_score_test)
ocsvm_ap  = average_precision_score(y_test, ocsvm_score_test)
print(f'   📈 ROC-AUC       (test): {ocsvm_auc:.4f}')
print(f'   📈 Avg Precision (test): {ocsvm_ap:.4f}')
print('✅ One-Class SVM done!')

# ── Per-model metrics (using best_thresh from ensemble — set default 0.5 for individual eval)
_thresh_ind = 0.5
_pred = (ocsvm_score_test >= _thresh_ind).astype(int)
_acc  = accuracy_score(y_test, _pred)
_rec  = recall_score(y_test, _pred, zero_division=0)
_f1   = f1_score(y_test, _pred, zero_division=0)
_auc  = roc_auc_score(y_test, ocsvm_score_test)
_ap   = average_precision_score(y_test, ocsvm_score_test)
print(f'\n   ┌── Individual Model Metrics (threshold=0.5) ──────')
print(f'   │  Accuracy  : {_acc*100:.2f}%')
print(f'   │  Recall    : {_rec:.4f}')
print(f'   │  F1-Score  : {_f1:.4f}')
print(f'   │  ROC-AUC   : {_auc:.4f}')
print(f'   │  Avg Prec  : {_ap:.4f}')
print(f'   └───────────────────────────────────────────────────')


## 🏆 Cell 14 — Weighted Ensemble Risk Score

In [ ]:
print('🏆 Computing Weighted Ensemble Risk Score...')
print('   Semi-supervised: LP · LS · ST-RF · ST-ET')
print('   Unsupervised   : IsolationForest · OneClassSVM')
print('   Weights from labelled val AP — no test leakage')

lab_X     = X_train_scaled[lab_idx]
lab_y     = y_train[lab_idx]
lab_X_pca = pca.transform(lab_X)

val_lp    = np.nan_to_num(lp_model.predict_proba(lab_X_pca)[:, 1], nan=0.0)
val_ls    = np.nan_to_num(ls_model.predict_proba(lab_X_pca)[:, 1], nan=0.0)
val_st    = st_model.predict_proba(lab_X)[:, 1]
val_et    = et_model.predict_proba(lab_X)[:, 1]
val_iso_r = -iso.decision_function(lab_X)
val_iso   = ((val_iso_r   - iso_min)   / (iso_max   - iso_min   + 1e-9)).clip(0, 1)
val_ocs_r = -ocsvm.decision_function(lab_X)
val_ocsvm = ((val_ocs_r   - ocsvm_min) / (ocsvm_max - ocsvm_min + 1e-9)).clip(0, 1)

def safe_ap(y, s):
    try:    return max(average_precision_score(y, s), 0.01)
    except: return 0.01

w_lp    = safe_ap(lab_y, val_lp)
w_ls    = safe_ap(lab_y, val_ls)
w_st    = safe_ap(lab_y, val_st)
w_et    = safe_ap(lab_y, val_et)
w_if    = safe_ap(lab_y, val_iso)
w_ocsvm = safe_ap(lab_y, val_ocsvm)
w_tot   = w_lp + w_ls + w_st + w_et + w_if + w_ocsvm

print(f'\n   Weights:')
print(f'     LP    : {w_lp:.3f}')
print(f'     LS    : {w_ls:.3f}')
print(f'     ST-RF : {w_st:.3f}')
print(f'     ST-ET : {w_et:.3f}')
print(f'     IsoF  : {w_if:.3f}')
print(f'     OCSVM : {w_ocsvm:.3f}')

def ensemble(lp, ls, st, et, iso_s, ocs):
    return (w_lp*lp + w_ls*ls + w_st*st + w_et*et + w_if*iso_s + w_ocsvm*ocs) / w_tot

risk_test  = ensemble(lp_proba_test,  ls_proba_test,  st_proba_test,  et_proba_test,  iso_score_test,  ocsvm_score_test)
risk_train = ensemble(lp_proba_train, ls_proba_train, st_proba_train, et_proba_train, iso_score_train, ocsvm_score_train)

ens_auc = roc_auc_score(y_test, risk_test)
ens_ap  = average_precision_score(y_test, risk_test)
print(f'\n   🎯 Ensemble ROC-AUC       : {ens_auc:.4f}')
print(f'   🎯 Ensemble Avg Precision : {ens_ap:.4f}')

# Threshold tuning on labelled val (no test leakage)
val_risk = ensemble(val_lp, val_ls, val_st, val_et, val_iso, val_ocsvm)
prec_v, rec_v, thresh_v = precision_recall_curve(lab_y, val_risk)
f1_v = 2 * prec_v * rec_v / (prec_v + rec_v + 1e-9)
best_thresh = float(np.clip(thresh_v[np.argmax(f1_v[:-1])], 0.0, 1.0))
print(f'\n   🎯 Best threshold (labelled val): {best_thresh:.4f}')

y_pred_test = (risk_test >= best_thresh).astype(int)
best_f1  = f1_score(y_test, y_pred_test)
accuracy = accuracy_score(y_test, y_pred_test)

print()
print('=' * 62)
print('  CLASSIFICATION REPORT — HELD-OUT TEST SET')
print('=' * 62)
print(classification_report(y_test, y_pred_test,
                             target_names=['Normal', 'Anomaly']))
print('=' * 62)
print(f'  ✅  Overall Accuracy : {accuracy*100:.2f}%')
print(f'  ✅  ROC-AUC          : {ens_auc:.4f}')
print(f'  ✅  Avg Precision    : {ens_ap:.4f}')
print(f'  ✅  F1-Score         : {best_f1:.4f}')
print('=' * 62)


## 🚨 Cell 15 — Alert Generation

In [ ]:
ALERT_THRESHOLD = 0.55

print(f'🚨 Generating alerts on ALL data (alert threshold = {ALERT_THRESHOLD})...')

df_full_fe, _ = engineer(df_raw, fit_objects=FIT)
for col in FEATURE_COLS:
    if col not in df_full_fe.columns:
        df_full_fe[col] = 0.0

X_full_imp    = imputer.transform(df_full_fe[FEATURE_COLS].values)
X_full_scaled = scaler.transform(X_full_imp)
X_full_pca    = pca.transform(X_full_scaled)

lp_full    = np.nan_to_num(lp_model.predict_proba(X_full_pca)[:, 1], nan=0.0)
ls_full    = np.nan_to_num(ls_model.predict_proba(X_full_pca)[:, 1], nan=0.0)
st_full    = st_model.predict_proba(X_full_scaled)[:, 1]
et_full    = et_model.predict_proba(X_full_scaled)[:, 1]
iso_r_full = -iso.decision_function(X_full_scaled)
iso_full   = ((iso_r_full   - iso_min)   / (iso_max   - iso_min   + 1e-9)).clip(0, 1)
ocs_r_full = -ocsvm.decision_function(X_full_scaled)
ocsvm_full = ((ocs_r_full   - ocsvm_min) / (ocsvm_max - ocsvm_min + 1e-9)).clip(0, 1)
risk_full  = ensemble(lp_full, ls_full, st_full, et_full, iso_full, ocsvm_full)

df_result = df_raw.copy().reset_index(drop=True)
df_result['risk_score']  = risk_full
df_result['is_anomaly']  = (risk_full >= best_thresh).astype(int)
df_result['alert_level'] = pd.cut(
    risk_full,
    bins=[0.0, 0.35, 0.55, 0.75, 1.001],
    labels=['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'],
    include_lowest=True
)

alerts = df_result[df_result['risk_score'] >= ALERT_THRESHOLD].copy()
alerts = alerts.sort_values('risk_score', ascending=False)

print(f'\n   Total logins       : {len(df_result):,}')
print(f'   Anomalies flagged  : {df_result["is_anomaly"].sum():,}')
print(f'   Alerts generated   : {len(alerts):,}  ({len(alerts)/len(df_result):.1%} of dataset)')
print(f'\n   Alert Breakdown:')
print(alerts['alert_level'].value_counts().to_string())

show_cols = [c for c in [
    'User ID', 'Country', 'Device Type', 'Login Successful',
    'Is Attack IP', 'Is Account Takeover', 'risk_score', 'alert_level'
] if c in alerts.columns]

print('\n🔴 Top 10 Highest-Risk Logins:')
display(alerts[show_cols].head(10))

try:
    alerts[show_cols].to_csv('/content/drive/MyDrive/rba_alerts.csv', index=False)
    print('\n✅ Alerts saved → Google Drive')
except:
    alerts[show_cols].to_csv('/content/rba_alerts.csv', index=False)
    print('\n✅ Alerts saved → /content/rba_alerts.csv')


## 📊 Cell 16 — Full Visualisation Dashboard

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 3, figsize=(19, 12))
fig.suptitle('RBA Semi-Supervised Anomaly Detection — Results Dashboard',
             fontsize=15, fontweight='bold', y=1.01)

# (A) Risk Score Distribution
ax = axes[0, 0]
ax.hist(risk_test[y_test == 0], bins=40, alpha=0.65, label='Normal',  color='steelblue', density=True)
ax.hist(risk_test[y_test == 1], bins=40, alpha=0.65, label='Anomaly', color='crimson',   density=True)
ax.axvline(best_thresh,      color='gold',   lw=2.5, ls='--', label=f'Threshold={best_thresh:.2f}')
ax.axvline(ALERT_THRESHOLD,  color='orange', lw=2,   ls=':',  label=f'Alert thr={ALERT_THRESHOLD}')
ax.set_title('Ensemble Risk Score — Test Set', fontweight='bold')
ax.set_xlabel('Risk Score'); ax.set_ylabel('Density'); ax.legend(fontsize=8)

# (B) ROC Curves
ax = axes[0, 1]
for name, scores, color in [
    ('Label Propagation', lp_proba_test,    'royalblue'),
    ('Label Spreading',   ls_proba_test,    'mediumorchid'),
    ('ST-RF',             st_proba_test,    'seagreen'),
    ('ST-ET',             et_proba_test,    'teal'),
    ('IsolationForest',   iso_score_test,   'darkorange'),
    ('OneClassSVM',       ocsvm_score_test, 'sienna'),
    ('Ensemble ⭐',       risk_test,        'crimson'),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    lw  = 3 if 'Ensemble' in name else 1.2
    ax.plot(fpr, tpr, lw=lw, color=color, label=f'{name}  (AUC={auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_title('ROC Curves — Test Set', fontweight='bold')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=7)

# (C) Precision-Recall Curves
ax = axes[0, 2]
for name, scores, color in [
    ('Label Propagation', lp_proba_test,    'royalblue'),
    ('Label Spreading',   ls_proba_test,    'mediumorchid'),
    ('ST-RF',             st_proba_test,    'seagreen'),
    ('ST-ET',             et_proba_test,    'teal'),
    ('IsolationForest',   iso_score_test,   'darkorange'),
    ('OneClassSVM',       ocsvm_score_test, 'sienna'),
    ('Ensemble ⭐',       risk_test,        'crimson'),
]:
    p, r, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    lw = 3 if 'Ensemble' in name else 1.2
    ax.plot(r, p, lw=lw, color=color, label=f'{name}  (AP={ap:.3f})')
ax.set_title('Precision-Recall — Test Set', fontweight='bold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.legend(fontsize=7)

# (D) Confusion Matrix
ax = axes[1, 0]
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'],
            ax=ax, annot_kws={'size': 14})
ax.set_title(f'Confusion Matrix (thr={best_thresh:.2f})', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# (E) Feature Importances from Self-Training RF
ax = axes[1, 1]
fi = st_model.estimator_.feature_importances_
fi_series = (pd.Series(fi, index=FEATURE_COLS)
               .sort_values(ascending=True).tail(15))
colors_fi = ['crimson' if v >= fi_series.quantile(0.75) else 'steelblue'
             for v in fi_series.values]
fi_series.plot.barh(ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Top Feature Importances (ST-RF)', fontweight='bold')
ax.set_xlabel('Importance')

# (F) Alert Level Breakdown
ax = axes[1, 2]
alert_counts = df_result['alert_level'].value_counts()
colors_map   = {'LOW': 'steelblue', 'MEDIUM': 'orange',
                'HIGH': 'tomato',   'CRITICAL': 'darkred'}
wedge_colors = [colors_map.get(str(k), 'grey') for k in alert_counts.index]
ax.pie(alert_counts, labels=alert_counts.index, autopct='%1.1f%%',
       colors=wedge_colors, startangle=140, pctdistance=0.80)
ax.set_title('Alert Level Breakdown (Full Dataset)', fontweight='bold')

plt.tight_layout()
PLOT_PATH = '/content/rba_anomaly_dashboard.png'
plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Dashboard saved → {PLOT_PATH}')
try:
    import shutil
    shutil.copy(PLOT_PATH, '/content/drive/MyDrive/rba_anomaly_dashboard.png')
    print('✅ Also saved to Google Drive.')
except:
    pass


## 💾 Cell 17 — Save All Models

In [ ]:
MODELS_DIR = '/content/drive/MyDrive/rba_models/'
try:
    os.makedirs(MODELS_DIR, exist_ok=True)
except:
    MODELS_DIR = '/content/rba_models/'
    os.makedirs(MODELS_DIR, exist_ok=True)
    print('ℹ️  Drive not mounted — saving locally.')

joblib.dump(lp_model,  MODELS_DIR + 'label_propagation.pkl')
joblib.dump(ls_model,  MODELS_DIR + 'label_spreading.pkl')
joblib.dump(st_model,  MODELS_DIR + 'self_training_rf.pkl')
joblib.dump(et_model,  MODELS_DIR + 'self_training_et.pkl')
joblib.dump(iso,       MODELS_DIR + 'isolation_forest.pkl')
joblib.dump(ocsvm,     MODELS_DIR + 'one_class_svm.pkl')
joblib.dump(imputer,   MODELS_DIR + 'imputer.pkl')
joblib.dump(scaler,    MODELS_DIR + 'scaler.pkl')
joblib.dump(pca,       MODELS_DIR + 'pca.pkl')
joblib.dump(FIT,       MODELS_DIR + 'fit_objects.pkl')
joblib.dump({
    'best_thresh'    : best_thresh,
    'alert_thresh'   : ALERT_THRESHOLD,
    'weights'        : (w_lp, w_ls, w_st, w_et, w_if, w_ocsvm),
    'features'       : FEATURE_COLS,
    'iso_min_max'    : (iso_min,   iso_max),
    'ocsvm_min_max'  : (ocsvm_min, ocsvm_max),
}, MODELS_DIR + 'ensemble_meta.pkl')

print(f'✅ All models saved to: {MODELS_DIR}')


## 🔮 Cell 18 — Inference Function (Score New Logins)

In [ ]:
def predict_risk(df_new: pd.DataFrame) -> pd.DataFrame:
    """Score a batch of new login rows. Returns df with risk_score, is_anomaly, alert_level."""
    _meta  = joblib.load(MODELS_DIR + 'ensemble_meta.pkl')
    _fit   = joblib.load(MODELS_DIR + 'fit_objects.pkl')
    _imp   = joblib.load(MODELS_DIR + 'imputer.pkl')
    _scl   = joblib.load(MODELS_DIR + 'scaler.pkl')
    _pca   = joblib.load(MODELS_DIR + 'pca.pkl')
    _lp    = joblib.load(MODELS_DIR + 'label_propagation.pkl')
    _ls    = joblib.load(MODELS_DIR + 'label_spreading.pkl')
    _st    = joblib.load(MODELS_DIR + 'self_training_rf.pkl')
    _et    = joblib.load(MODELS_DIR + 'self_training_et.pkl')
    _iso   = joblib.load(MODELS_DIR + 'isolation_forest.pkl')
    _ocs   = joblib.load(MODELS_DIR + 'one_class_svm.pkl')

    feats  = _meta['features']
    wl, wls, ws, we, wi, wo = _meta['weights']
    wt     = wl + wls + ws + we + wi + wo
    thr    = _meta['best_thresh']
    imin, imax   = _meta['iso_min_max']
    omin, omax   = _meta['ocsvm_min_max']

    df_fe, _ = engineer(df_new.copy(), fit_objects=_fit)
    for col in feats:
        if col not in df_fe.columns:
            df_fe[col] = 0.0

    X  = _imp.transform(df_fe[feats].values)
    Xs = _scl.transform(X)
    Xp = _pca.transform(Xs)

    lp_p   = np.nan_to_num(_lp.predict_proba(Xp)[:, 1], nan=0.0)
    ls_p   = np.nan_to_num(_ls.predict_proba(Xp)[:, 1], nan=0.0)
    st_p   = _st.predict_proba(Xs)[:, 1]
    et_p   = _et.predict_proba(Xs)[:, 1]
    iso_p  = ((-_iso.decision_function(Xs) - imin) / (imax  - imin  + 1e-9)).clip(0, 1)
    ocs_p  = ((-_ocs.decision_function(Xs) - omin) / (omax  - omin  + 1e-9)).clip(0, 1)

    risk = (wl*lp_p + wls*ls_p + ws*st_p + we*et_p + wi*iso_p + wo*ocs_p) / wt

    out = df_new.copy().reset_index(drop=True)
    out['risk_score']  = risk
    out['is_anomaly']  = (risk >= thr).astype(int)
    out['alert_level'] = pd.cut(risk, bins=[0.0, 0.35, 0.55, 0.75, 1.001],
                                 labels=['LOW','MEDIUM','HIGH','CRITICAL'],
                                 include_lowest=True)
    return out

sample = df_raw.sample(5, random_state=99).reset_index(drop=True)
scored = predict_risk(sample)
show   = [c for c in ['User ID','Country','Is Attack IP','risk_score','is_anomaly','alert_level']
          if c in scored.columns]
print('🔮 Sample Predictions:')
display(scored[show].round(4))


## 📋 Cell 19 — Final Summary

In [ ]:
from sklearn.metrics import recall_score

print('=' * 70)
print('  ✅  FINAL RESULTS SUMMARY — All Models (Held-Out Test Set)')
print('=' * 70)
print(f'  Dataset rows      : {len(df_raw):,}')
print(f'  Train rows        : {len(df_train_raw):,}')
print(f'  Test  rows        : {len(df_test_raw):,}')
print(f'  Labelled fraction : {LABEL_RATIO:.0%}')
print(f'  Features used     : {len(FEATURE_COLS)}')
print()
print(f'  {"Model":<24}  {"Accuracy":>9}  {"Recall":>8}  {"F1":>7}  {"ROC-AUC":>8}  {"AvgPrec":>8}')
print(f'  {"-"*65}')

_thr = 0.5   # individual model threshold for accuracy/recall/f1

def model_row(name, scores):
    pred = (scores >= _thr).astype(int)
    acc  = accuracy_score(y_test, pred)
    rec  = recall_score(y_test, pred, zero_division=0)
    f1_  = f1_score(y_test, pred, zero_division=0)
    auc  = roc_auc_score(y_test, scores)
    ap   = average_precision_score(y_test, scores)
    print(f'  {name:<24}  {acc*100:>8.2f}%  {rec:>8.4f}  {f1_:>7.4f}  {auc:>8.4f}  {ap:>8.4f}')

model_row('Label Propagation',   lp_proba_test)
model_row('Label Spreading',     ls_proba_test)
model_row('Self-Training RF',    st_proba_test)
model_row('Self-Training ET',    et_proba_test)
model_row('Isolation Forest',    iso_score_test)
model_row('One-Class SVM',       ocsvm_score_test)

print(f'  {"-"*65}')
ens_pred = (risk_test >= best_thresh).astype(int)
ens_acc  = accuracy_score(y_test, ens_pred)
ens_rec  = recall_score(y_test, ens_pred, zero_division=0)
ens_f1   = f1_score(y_test, ens_pred, zero_division=0)
print(f'  {"Ensemble ⭐ (best thr)":<24}  {ens_acc*100:>8.2f}%  {ens_rec:>8.4f}  {ens_f1:>7.4f}  {ens_auc:>8.4f}  {ens_ap:>8.4f}')
print()
print(f'  Ensemble threshold used : {best_thresh:.4f}  (tuned on labelled val, not test)')
print(f'  Alerts generated        : {len(alerts):,}  ({len(alerts)/len(df_result):.1%})')
print('=' * 70)
print()
print('  Full Classification Report (Ensemble):')
print(classification_report(y_test, ens_pred, target_names=['Normal', 'Anomaly']))
print('  🔒 All models strictly semi-supervised or unsupervised ✅')
print('=' * 70)
